Muhammad Hassan Khalid

21L-5692

In [3]:
from collections import deque

def is_valid(x, y, maze):
    return 0 <= x < len(maze) and 0 <= y < len(maze[0]) and maze[x][y] != '1'

def get_neighbors(x, y, maze):
    neighbors = []
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    for dx, dy in directions:
        new_x, new_y = x + dx, y + dy
        if is_valid(new_x, new_y, maze):
            neighbors.append((new_x, new_y))
    return neighbors

def dfs(maze):
    stack = [(0, 0)]
    visited = set()
    path = {}

    while stack:
        x, y = stack.pop()
        if maze[x][y] == 'G':
            result = []
            while (x, y) in path:
                result.append((x, y))
                x, y = path[(x, y)]
            result.append((0, 0))
            return result[::-1]

        if (x, y) not in visited:
            visited.add((x, y))
            for neighbor in get_neighbors(x, y, maze):
                if neighbor not in visited:
                    stack.append(neighbor)
                    path[neighbor] = (x, y)

    return -1

def bfs(maze):
    queue = deque([(0, 0)])
    visited = set()
    path = {}

    while queue:
        x, y = queue.popleft()
        if maze[x][y] == 'G':
            result = []
            while (x, y) in path:
                result.append((x, y))
                x, y = path[(x, y)]
            result.append((0, 0))
            return result[::-1]

        if (x, y) not in visited:
            visited.add((x, y))
            for neighbor in get_neighbors(x, y, maze):
                if neighbor not in visited:
                    queue.append(neighbor)
                    path[neighbor] = (x, y)

    return -1

from numpy import loadtxt
maze = loadtxt("2d.txt" , dtype = str)
print(maze)
print()

print("Depth-First Search" , dfs(maze))
print("Breadth-First Search" , bfs(maze))

[['S' '0' '0' '0' '1' '0' '0']
 ['1' '1' '0' '0' '0' '1' '1']
 ['0' '1' '0' '1' '0' '0' '0']
 ['1' '1' '0' '1' '1' '0' '1']
 ['0' '1' '0' '1' '0' '0' '0']
 ['0' '1' '1' '1' '0' '1' '1']
 ['G' '0' '0' '0' '0' '0' '0']]

Depth-First Search [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (1, 4), (2, 4), (2, 5), (3, 5), (4, 5), (4, 4), (5, 4), (6, 4), (6, 3), (6, 2), (6, 1), (6, 0)]
Breadth-First Search [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (1, 4), (2, 4), (2, 5), (3, 5), (4, 5), (4, 4), (5, 4), (6, 4), (6, 3), (6, 2), (6, 1), (6, 0)]


In [5]:
import heapq

class Graph:
    def __init__(self):
        self.graph = {}

    def add_edge(self, node1, node2, weight):
        if node1 not in self.graph:
            self.graph[node1] = []
        if node2 not in self.graph:
            self.graph[node2] = []
        self.graph[node1].append((node2, weight))
        self.graph[node2].append((node1, weight))  # Assuming undirected graph

    def ucs(self, start, goal):
        visited = set()
        heap = [(0, start, [])]  # (cost, node, path)

        while heap:
            cost, node, path = heapq.heappop(heap)
            if node not in visited:
                visited.add(node)
                path = path + [node]

                if node == goal:
                    return path, cost

                for neighbor, weight in self.graph[node]:
                    if neighbor not in visited:
                        heapq.heappush(heap, (cost + weight, neighbor, path))

        return None, float('inf')

# Example usage:
g = Graph()
g.add_edge('A', 'B', 4)
g.add_edge('A', 'C', 2)
g.add_edge('B', 'C', 5)
g.add_edge('B', 'D', 10)
g.add_edge('C', 'D', 3)

start_node = 'A'
goal_node = 'D'
shortest_path, cost = g.ucs(start_node, goal_node)

if shortest_path:
    print(f"Shortest path from {start_node} to {goal_node}: {shortest_path}")
    print(f"Cost: {cost}")
else:
    print(f"No path found from {start_node} to {goal_node}")


Shortest path from A to D: ['A', 'C', 'D']
Cost: 5


In [15]:
class CubeSolver:
    def __init__(self, cube):
        self.cube = cube
        self.rows = len(self.cube)
        self.cols = len(self.cube[0])
        self.start = None
        self.goal = None
        self.initialize()

    def initialize(self):
        for i in range(self.rows):
            for j in range(self.cols):
                if self.cube[i][j] == 'S':
                    self.start = (i, j)
                elif self.cube[i][j] == 'G':
                    self.goal = (i, j)

    def heuristic(self, node):
        x, y = node
        if x < 0 or x >= self.rows or y < 0 or y >= self.cols or self.cube[x][y] == '+':
            return float('inf')  # Out of bounds or short wall, return infinity
        return abs(x - self.goal[0]) + abs(y - self.goal[1])

    def get_neighbors(self, node):
        x, y = node
        neighbors = []
        for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            new_x, new_y = x + dx, y + dy
            if 0 <= new_x < self.rows and 0 <= new_y < self.cols and self.cube[new_x][new_y] != '1':
                neighbors.append((new_x, new_y))
        return neighbors

    def solve(self):
        visited = set()
        priority_queue = [(0, self.start, [])]  # (f-score, node, path)
        while priority_queue:
            _, current, path = min(priority_queue)
            priority_queue.remove((_, current, path))
            if current == self.goal:
                return path
            if current in visited:
                continue
            visited.add(current)
            for neighbor in self.get_neighbors(current):
                if neighbor not in visited:
                    new_path = path + [neighbor]
                    f_score = len(new_path) + self.heuristic(neighbor)
                    priority_queue.append((f_score, neighbor, new_path))
        return None

def read_cube_from_file(filename):
    cube = []
    with open(filename, 'r') as f:
        for line in f:
            cube.append(line.strip().split())
    return cube

# Read cube from file
cube_data = read_cube_from_file('2d.txt')

# Solve the cube
solver = CubeSolver(cube_data)
path = solver.solve()

# Print the result
if path:
    print("Shortest path:")
    for row, col in path:
        print(f"({row}, {col})")
else:
    print("No path found.")


Shortest path:
(1, 2)
(2, 2)
(3, 2)
(4, 2)
(4, 3)
(4, 4)
(5, 4)
(6, 4)
(6, 3)
(6, 2)
(6, 1)
(6, 0)
